In [8]:
import json
from pathlib import Path

DATA_DIR = Path("../data")
PRED_PATH = DATA_DIR / "moduleB_output.json"
GOLD_PATH = DATA_DIR / "gold.json"

In [9]:
with open(PRED_PATH, "r") as f:
    preds_data = json.load(f)

with open(GOLD_PATH, "r") as f:
    gold_data = json.load(f)

print("Loaded:")
print("Pred cases:", len(preds_data))
print("Gold cases:", len(gold_data))

Loaded:
Pred cases: 10
Gold cases: 10


In [10]:
pred_map = {c["case_id"]: c for c in preds_data}
gold_map = {c["case_id"]: c for c in gold_data}

case_ids = list(pred_map.keys())
case_ids

['CASE_001',
 'CASE_002',
 'CASE_003',
 'CASE_004',
 'CASE_005',
 'CASE_006',
 'CASE_007',
 'CASE_008',
 'CASE_009',
 'CASE_010']

In [11]:
def score_case(case_id):
    pred_case = pred_map[case_id]
    gold_case = gold_map[case_id]

    pred = {m["surface"].lower().strip()
            for m in pred_case.get("excluded_mentions", [])}

    gold = {g.lower().strip()
            for g in gold_case.get("excluded", [])}

    tp = len(pred & gold)
    fp = len(pred - gold)
    fn = len(gold - pred)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) else 0.0)

    return {
        "case_id": case_id,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }

In [12]:
import pandas as pd

results = [score_case(cid) for cid in case_ids]
df = pd.DataFrame(results)
df

,case_id,TP,FP,FN,Precision,Recall,F1
0,CASE_001,6,6,11,0.500000,0.352941,0.413793
1,CASE_002,2,2,6,0.500000,0.250000,0.333333
2,CASE_003,2,1,6,0.666667,0.250000,0.363636
3,CASE_004,7,1,1,0.875000,0.875000,0.875000
4,CASE_005,3,5,4,0.375000,0.428571,0.400000
5,CASE_006,1,1,4,0.500000,0.200000,0.285714
6,CASE_007,2,0,2,1.000000,0.500000,0.666667
7,CASE_008,5,3,4,0.625000,0.555556,0.588235
8,CASE_009,5,1,2,0.833333,0.714286,0.769231
9,CASE_010,4,1,2,0.800000,0.666667,0.727273


In [13]:
import numpy as np

# --- Macro averages (simple mean over cases)
macro_precision = df["Precision"].mean()
macro_recall = df["Recall"].mean()
macro_f1 = df["F1"].mean()

# --- Micro totals (sum TP/FP/FN over all cases)
TP_sum = df["TP"].sum()
FP_sum = df["FP"].sum()
FN_sum = df["FN"].sum()

micro_precision = TP_sum / (TP_sum + FP_sum) if (TP_sum + FP_sum) else 0.0
micro_recall = TP_sum / (TP_sum + FN_sum) if (TP_sum + FN_sum) else 0.0
micro_f1 = (
    2 * micro_precision * micro_recall / (micro_precision + micro_recall)
    if (micro_precision + micro_recall) else 0.0
)

print("=== Macro (per-case average) ===")
print(f"Precision_macro: {macro_precision:.3f}")
print(f"Recall_macro:    {macro_recall:.3f}")
print(f"F1_macro:        {macro_f1:.3f}")

print("\n=== Micro (pooled over all cases) ===")
print(f"Precision_micro: {micro_precision:.3f}")
print(f"Recall_micro:    {micro_recall:.3f}")
print(f"F1_micro:        {micro_f1:.3f}")

=== Macro (per-case average) ===
Precision_macro: 0.667
Recall_macro:    0.479
F1_macro:        0.542

=== Micro (pooled over all cases) ===
Precision_micro: 0.638
Recall_micro:    0.468
F1_micro:        0.540
